In [1]:
#### mv utils ####
import modules.data as d
import modules.model as m
import modules.train as t
import pyg_modules.utils as u

#### packages ####
import torch
import torch.nn as nn
from torch_geometric.data import InMemoryDataset, Data

import pandas as pd
import numpy as np
import os
from pathlib import Path

from sklearn.preprocessing import RobustScaler, StandardScaler
from modules.model import _get_edge_index
from pyg_modules.utils import vprint, dict_summary

#### typing ####
from torch import Tensor
from pandas import DataFrame
from typing import Literal

In [2]:
#### init data ####

# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device(['cuda:1', 'cuda:0'])

*** Device() ***
device = cuda:2



---

In [3]:
class Preprocessor():
    def __init__(
        self, 
        # paths
        tcga_project:str, 
        tcga_dir:str, 
        relation_filepath:str, 
        metadata_subtype_col:str='', 

        # fc
        apply_fold_change:bool=False,
        metadata_ctrl:str='Normal',

        # sample preprocessing
        log0_method:Literal['log1p','offset']='log1p',
        apply_DESeq_norm:bool=True, 
        log_transform:bool=True,
        scale_method:Literal['robust', 'standard', None]=None,

        # filter, resample, etc.
        y_col:str='type',
        y_format:Literal['index','onehot']='index',
        drop:list[str]=None,
        max_subset:int=None,
        edge_method:Literal['in','out']='in',
        verbose:bool=True,
    ):
        '''
        Class to format and preprocess TCGA data, TCGA metadata, and KEGG relation data.
        '''
        
        vprint('# #### Preprocessor() ####', verbose=verbose)
        
        # assign params to inst vars
        self.log0_method = log0_method

        # read files into df
        gene_counts = self._read_gene_counts(tcga_project, tcga_dir)
        metadata = self._read_metadata(tcga_project, tcga_dir, metadata_subtype_col)
        relation = pd.read_csv(relation_filepath)

        # data preprocessing
        if apply_DESeq_norm == True:
            gene_counts = self._DESeq_norm(gene_counts)
            self.deseq = gene_counts

        if apply_fold_change == True:
            gene_counts = self._fold_change(gene_counts, metadata, metadata_ctrl)

        if log_transform == True:
            gene_counts = np.log(self._handle_log0(gene_counts))

        if scale_method != None:
            gene_counts = self._scale_gene_counts(gene_counts, scale_method)

        # preprocess gene_counts and relation into graph data
        gene_counts, relation, node_id_map = self._graph_preprocessing(gene_counts, relation)

        # get masks, flatten relation
        masks = self._get_masks(relation)
        relation = relation.drop(columns='pathway_name').groupby(['idx1','idx2'], as_index=False).any()

        # filter counts by class (drop classes, downsample) if applicable
        gene_counts, metadata = self._filter_counts(gene_counts, metadata, y_col, drop, max_subset)

        # get xy
        x, y, y_labels = self._get_xy(gene_counts, metadata, y_col, y_format)

        # get class_weights
        self.class_weights = y.shape[0]/y.sum(dim=0)

        # get edge information
        self.edge_index, self.edge_attr = self._get_edges(relation, edge_method)

        # assign instance variables
        self.gene_counts = gene_counts
        self.metadata = metadata
        self.relation = relation
        self.node_id_map = node_id_map
        self.masks = masks
        self.x = x
        self.y = y
        self.y_labels = y_labels

        # get dims
        self._get_dims(y_format)
        vprint(self, verbose=verbose)

    def _read_gene_counts(self, tcga_project:str, tcga_dir:str):
        # read df
        df = pd.read_csv(Path(tcga_dir) / f'{tcga_project}_gene_counts.csv')

        # rename 'Unnamed: 0' col if applicable
        if 'Unnamed: 0' in df.columns:
            df = df.rename(columns={'Unnamed: 0':'ensg'})

        # remove ensg version
        df['ensg'] = df['ensg'].str.split('.').str[0]

        # set index, column name
        df = df.set_index('ensg')
        df.columns.name = 'barcode'

        return df

    def _read_metadata(self, tcga_project:str, tcga_dir:str, metadata_subtype_col:str):
        # read df
        df_complete = pd.read_csv(Path(tcga_dir) / f'{tcga_project}_metadata.csv')

        # drop 'Unnamed: 0' col if applicable
        if 'Unnamed: 0' in df_complete.columns:
                df_complete = df_complete.drop(columns='Unnamed: 0')

        # compile df
        df = pd.DataFrame(
            {
                'barcode': df_complete['barcode'],
                'type': df_complete.apply(lambda row: row['name'] if row['tissue_type'] == 'Tumor' else row['tissue_type'], axis=1),
            }
        )

        # append subtype if applicable
        if metadata_subtype_col != '':
            df['subtype'] = df_complete[metadata_subtype_col].fillna(df_complete['sample_type'])

        return df
    
    def _fold_change(self, gene_counts:pd.DataFrame, metadata:pd.DataFrame, metadata_ctrl:str):
        # get control barcodes
        ctrl_barcodes = metadata[metadata['type'] == metadata_ctrl]['barcode']

        # get mean
        ctrl_avg = gene_counts[ctrl_barcodes].mean(axis=1).values.reshape(-1, 1) # reshape to allow division

        # get fc
        return gene_counts / ctrl_avg
    
    def _handle_log0(self, x):
        # offset (replace 0 with small number)
        if self.log0_method == 'offset':
            if type(x) == pd.DataFrame:
                x = x.replace(0, 1e-6)
            else:
                x = x + 1e-6 # if x is not a df, add 1e-6

        # log1p method (add 1 before log)
        elif self.log0_method == 'log1p':
            x = x + 1

        # error msg
        else:
            print("Invalid log0 method; log0 method not applied. Use 'offset' or 'log1p'.")

        return x

    def _DESeq_norm(self, gene_counts:pd.DataFrame):
        # handle log zero
        gene_counts = self._handle_log0(gene_counts)

        # take the (natural) log of all the values
        log_counts = np.log(gene_counts)

        # average each row (e.g., geometric average)
        geom_avg = log_counts.mean(axis=1)

        # filter out +-inf
        geom_avg_filt = geom_avg[(abs(geom_avg) != np.inf)]

        # subtract the average log value from the log(counts)
        log_ratio = log_counts.sub(geom_avg_filt, axis=0)

        # caclulate the median of the ratios for each sample
        log_ratio_median = log_ratio.median()

        # calculate scaling factors (e^log_ratio_median)
        scaling_factor = log_ratio_median.apply(lambda x: np.exp(x))

        # divide the original read counts by the scaling factors; return output
        return gene_counts.div(scaling_factor, axis=1)
    
    def _scale_gene_counts(self, gene_counts:pd.DataFrame, scale_method:Literal['robust', 'standard', None]=None):
        # return if none
        if scale_method == None:
            return gene_counts

        # select scaler        
        if scale_method == 'robust':
            scaler = RobustScaler()
        elif scale_method == 'standard':
            scaler = StandardScaler()

        # scale data
        scaled_values = scaler.fit_transform(gene_counts.T).T

        # convert to df
        gene_counts = pd.DataFrame(
            scaled_values,
            index=gene_counts.index,
            columns=gene_counts.columns
        )

        return gene_counts
    
    def _graph_preprocessing(self, gene_counts:pd.DataFrame, relation:pd.DataFrame):
        # filter gene_counts by relation ensembl
        unique_ensg = pd.concat([relation[cols] for cols in ['ensembl1', 'ensembl2']]).unique().tolist()
        gene_counts = gene_counts.loc[unique_ensg,:]

        # get id maps
        node_id_map = {node: int(i) for i, node in enumerate(gene_counts.index)}

        # map relation ensembl id to nodes
        relation['idx1'] = relation['ensembl1'].map(node_id_map)
        relation['idx2'] = relation['ensembl2'].map(node_id_map)

        # replace ensembl with idx
        cols = relation.columns.to_list()
        cols = [col for col in cols if col not in ['pathway_name', 'ensembl1', 'ensembl2','idx1','idx2']]
        cols = ['pathway_name', 'idx1', 'idx2'] + cols
        relation = relation.loc[:,cols]

        return gene_counts, relation, node_id_map

    def _get_masks(self, relation:pd.DataFrame):
        # initialize empty dict
        mask_nodes = {}

        # iterate through df grouped by mask_id
        for mask_id, group in relation.groupby('pathway_name'):
            nodes = pd.concat([group['idx1'], group['idx2']]).unique() # get unique nodes in idx1 & idx2
            mask_nodes[mask_id] = nodes.tolist() # append to dict

        # list masks
        masks = [j for _,j in mask_nodes.items()]

        return masks
    
    def _filter_counts(self, gene_counts:pd.DataFrame, metadata:pd.DataFrame, y_col:str, drop:dict, max_subset:int):
            # drop cols by class
            if (drop != None) & (type(drop) == dict):
                for key, value in drop.items():
                    metadata = metadata[~metadata[key].isin(value)]

            # downsample
            if (max_subset != None) & (type(max_subset) == int):

                # helper fxn
                def downsample(group):
                    if len(group) > max_subset:
                        return group.sample(n=max_subset)
                    return group
                
                # apply downsampling
                metadata_grouped = metadata.groupby(y_col)
                metadata_grouped = metadata_grouped.apply(downsample, include_groups=False).reset_index(drop=True)

                # reappend y_col, reorder
                metadata = pd.merge(metadata_grouped, metadata[['barcode', y_col]], on='barcode', how='left')
                metadata = metadata[['barcode','type','subtype']]

            # apply filter
            gene_counts = gene_counts[metadata['barcode']]

            return gene_counts, metadata

    def _get_xy(self, gene_counts:pd.DataFrame, metadata:pd.DataFrame, y_col:str, y_format:Literal['index','onehot']):
        # format x
        x = gene_counts.T
        x = x.values.astype(np.float32)
        x = np.expand_dims(x, axis=-1) # reshape to (num_samples, num_nodes, 1)
        x = torch.tensor(x)

        # format y
        y_labels = metadata[y_col]
        y = pd.get_dummies(y_labels).values.astype(np.float32)
        y_labels = y_labels.unique().tolist()        
        y = torch.tensor(y)

        # convert y to index, if applicable
        y = y.argmax(dim=1) if y_format == 'index' else y
        
        return x, y, y_labels

    def _get_edges(self, relation:DataFrame, edge_method:Literal['in','out'], source:str='idx1', target:str='idx2'):
        # get edge information >> (num_edges, 2)
        if edge_method == 'in':
            edge_index = relation[[target, source]]
        elif edge_method == 'out':
            edge_index = relation[[source, target]]
        
        # transpose, convert to tensor >> (2, num_edges) 
        edge_index = torch.tensor(edge_index.values.T, dtype=torch.int64)

        # get edge attr
        attr_cols = relation.columns.drop([source, target])
        if len(attr_cols) > 0:
            edge_attr = torch.tensor(relation[attr_cols].values, dtype=torch.int64)
        else:
            edge_attr = None

        return edge_index, edge_attr

    def _get_dims(self, y_format:Literal['index','onehot']):
        # x
        self.num_samples, self.num_nodes, self.num_node_features = self.x.shape

        # y
        if y_format == 'onehot':
            _, self.num_classes = self.y.shape
        else:
            self.num_classes = self.y.unique().size(0)

        # masks
        self.num_masks = len(self.masks)

        # edges
        self.num_edges, self.num_edge_features = self.edge_attr.shape
    
    def __str__(self):
        return dict_summary(self.__dict__)

In [4]:
brca = Preprocessor(
    tcga_project='TCGA-BRCA',
    tcga_dir=datasets/'tcga',
    relation_filepath=datasets/'other'/'relation_ohe.csv',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',
    
    # counts
    apply_DESeq_norm=True, 
    log_transform=True,
    scale_method=None,

    # etc
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Primary Tumor']},
    max_subset = 120,
)

# #### Preprocessor() ####
# log0_method              log1p                    str
# deseq                    (60660, 1231)            DataFrame
# class_weights            ()                       Tensor (cuda:2)
# edge_index               (2, 32798)               Tensor (cuda:2)
# edge_attr                (32798, 16)              Tensor (cuda:2)
# gene_counts              (4383, 562)              DataFrame
# metadata                 (562, 3)                 DataFrame
# relation                 (32798, 18)              DataFrame
# node_id_map              4383                     dict
# masks                    305                      list
# x                        (562, 4383, 1)           Tensor (cuda:2)
# y                        (562,)                   Tensor (cuda:2)
# y_labels                 6                        list
# num_samples              562                      int
# num_nodes                4383                     int
# num_node_features        1                  

---

In [5]:
class GraphDataset(InMemoryDataset):
    def __init__(self, preprocessor:Preprocessor, verbose:bool=True, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.x = preprocessor.x
        self.y = preprocessor.y
        self.edge_index = preprocessor.edge_index
        self.edge_attr = preprocessor.edge_attr        

        # process data
        self.data, self.slices = self._process_data()

        # print summary
        if verbose == True:
            self.print_dims()

    def _process_data(self): 
        data_list = []
        num_samples = self.x.size(0)

        # for each sample
        for i in range(num_samples):
            
            # create a graph (Data) per sample
            data_entry = Data(
                x=self.x[i],
                y=self.y[i],
                edge_index=self.edge_index,
                edge_attr=self.edge_attr,
            )

            # add sample id
            data_entry.sample_id = i

            # append to list
            data_list.append(data_entry)

        # collate list
        data, slices = self.collate(data_list)

        return data, slices

    def print_dict(self):
        # get first graph in dataset
        data = self[0] 

        # format msg
        out = '# #### GraphDataset(), Dataset (Dict) ####\n'
        out += dict_summary(self.__dict__)
        out += '\n# #### GraphDataset(), Data (Dict) ####\n'
        out += dict_summary(data.__dict__)

        # print msg
        print(out)

    def print_dims(self):
        # get first graph in dataset
        data = self[0]

        # get dims
        dataset_dims = {
            'num_graphs (len)':len(self),
            'num_node_features':self.num_node_features,
            'num_edge_features':self.num_edge_features,
        }
        data_dims = {
            'num_nodes':data.num_nodes,
            'num_edges':data.num_edges,
            'num_node_features':data.num_node_features,
            'num_edge_features':data.num_edge_features,
        }
        data_summary = {
            'Average node degree': data.num_nodes / data.num_edges,
            'Has isolated nodes': data.has_isolated_nodes(),
            'Has self-loops': data.has_self_loops(),
            'Directionality': 'directed' if data.is_directed() else 'undirected'
        }

        # create msg
        out = '# #### GraphDataset(), Dataset ####\n'
        out += dict_summary(dataset_dims)
        out += '\n# #### GraphDataset(), Data ####\n'
        out += dict_summary(data_dims)
        out += '\n# #### GraphDataset(), Summary ####\n'
        out += dict_summary(data_summary)

        # print msg
        print(out)
       

In [6]:
brca_graph = GraphDataset(brca)

# #### GraphDataset(), Dataset ####
# num_graphs (len)         562                      int
# num_node_features        1                        int
# num_edge_features        16                       int

# #### GraphDataset(), Data ####
# num_nodes                4383                     int
# num_edges                32798                    int
# num_node_features        1                        int
# num_edge_features        16                       int

# #### GraphDataset(), Summary ####
# Average node degree      0.1336                   float
# Has isolated nodes       True                     bool
# Has self-loops           True                     bool
# Directionality           directed                 str



---

In [33]:
import os
import torch
from torch_geometric.data import Dataset
from torch_geometric.utils import subgraph


In [ ]:
class SubgraphDataset(Dataset):
    def __init__(self, root, graph_dataset, subgraph_nodes, *args, **kwargs):
        super().__init__(root, *args, **kwargs)
        self.graph_dataset = graph_dataset
        self.subgraph_nodes = subgraph_nodes
        self.num_subgraphs = len(subgraph_nodes)
        
        # process data
        self.data, self.slices = self._process_data(graph_dataset, subgraph_nodes)

    @property
    def processed_file_names(self):
        # get list of filenames as data_(graph_idx)_(subgraph_idx).pt
        # note: graph_idx = sample_id; subgraph_idx = subgraph_id
        fname = [
            f'data_{i}_{j}.pt'
            for i in range(len(self.graph_dataset))
            for j in range(len(self.num_subgraphs))
        ]

        return fname

    def len(self):
        # num entries in dataset = (num graphs) * (num subgraphs per graph)
        num_entries = len(self.graph_dataset) * self.num_subgraphs
        return num_entries
    
    def get(self, idx):
        sample_id = idx // self.num_subgraphs
        



    def _process_data(self, dataset, subgraph_nodes):
        data_list = []

        for data in dataset:
            sample_id = data.sample_id

            for subgraph_id, node_ids in enumerate(subgraph_nodes):

                # convert node ids to tensor
                node_ids_tensor = torch.tensor(node_ids)

                # get subgraph information
                sub_edge_index, edge_mask = subgraph(
                    node_ids_tensor,
                    data.edge_index,
                    relabel_nodes=True
                )

                # create subgraph
                sub_data = Data(
                    x=data.x[node_ids_tensor],
                    y=data.y,
                    edge_index=sub_edge_index,
                    edge_attr=data.edge_attr[edge_mask],
                )

                # add ids
                sub_data.sample_id = sample_id
                sub_data.subgraph_id = subgraph_id

                # append to list
                data_list.append(sub_data)

        # collate data
        return self.collate(data_list)
                


In [ ]:
class DiskSubgraphDataset(Dataset):
    def __init__(self, root, full_dataset, subgraph_node_list, transform=None, pre_transform=None):
        self.full_dataset = full_dataset
        self.subgraph_node_list = subgraph_node_list
        self.k = len(subgraph_node_list)
        super().__init__(root, transform, pre_transform)

    @property
    def raw_file_names(self):
        return []  # Not using raw files here

    @property
    def processed_file_names(self):
        return [f"data_{i}_{j}.pt"
                for i in range(len(self.full_dataset))
                for j in range(self.k)]

    def len(self):
        return len(self.full_dataset) * self.k

    def get(self, idx):
        # idx = flat index over num_samples * k
        sample_id = idx // self.k
        subgraph_id = idx % self.k
        path = os.path.join(self.processed_dir, f"data_{sample_id}_{subgraph_id}.pt")
        data = torch.load(path)
        return data

    def process(self):
        os.makedirs(self.processed_dir, exist_ok=True)

        for i, sample in enumerate(self.full_dataset):
            for subgraph_id, node_ids in enumerate(self.subgraph_node_list):
                node_ids_tensor = torch.tensor(node_ids, dtype=torch.long)

                sub_edge_index, edge_mask = subgraph(
                    node_ids_tensor, sample.edge_index, relabel_nodes=True
                )

                sub_data = Data(
                    x=sample.x[node_ids_tensor],
                    edge_index=sub_edge_index,
                    edge_attr=sample.edge_attr[edge_mask],
                    y=sample.y,
                    sample_id=i,
                    subgraph_id=subgraph_id
                )

                path = os.path.join(self.processed_dir, f"data_{i}_{subgraph_id}.pt")
                torch.save(sub_data, path)


OutOfMemoryError: CUDA out of memory. Tried to allocate 670.18 GiB. GPU 5 has a total capacity of 79.15 GiB of which 67.55 GiB is free. Including non-PyTorch memory, this process has 11.58 GiB memory in use. Of the allocated memory 9.66 GiB is allocated by PyTorch, and 1.44 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---

In [7]:
# from torch_geometric.loader import DataLoader

# class QuickTrainer():
#     def __init__(self, dataset:GraphDataset, generator):
#         self.dataset = dataset
#         self.generator = generator

#     def _get_dataloaders(self, dataset:GraphDataset, batch_size:int=64, test_size:float=0.15):
#         dataset = dataset.shuffle()
#         num_samples = len(dataset)

#         # train-test split
#         num_train_samples = num_samples - round(num_samples * test_size)
#         train_dataset = dataset[:num_train_samples]
#         test_dataset = dataset[num_train_samples:]

#         # create dataloaders
#         train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=self.generator)
#         test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, generator=self.generator)

#         return train_loader, test_loader
    
#     def _train(self, model, loader, optimizer, criterion):
#         model.train()

#         for data in loader:  # Iterate in batches over the training dataset.
#             out = model(data.x, data.edge_index, data.batch)  # Perform a single forward pass.
#             loss = criterion(out, data.y)  # Compute the loss.
#             loss.backward()  # Derive gradients.
#             optimizer.step()  # Update parameters based on gradients.
#             optimizer.zero_grad()  # Clear gradients.

#     def _test(self, model, loader):
#         model.eval()

#         correct = 0
#         for data in loader:  # Iterate in batches over the training/test dataset.
#             out = model(data.x, data.edge_index, data.batch)  
#             pred = out.argmax(dim=1)  # Use the class with highest probability.
#             correct += int((pred == data.y).sum())  # Check against ground-truth labels.
#         return correct / len(loader.dataset)  # Derive ratio of correct predictions.
    
#     def get_toy_databatch(self, batch_size:int=64):
#         loader = DataLoader(self.dataset, batch_size=batch_size, shuffle=True, generator=self.generator)
#         batches = [(step, data) for step, data in enumerate(loader)]
#         return batches[0][1]

#     def train_model(self, model, batch_size:int=64, test_size:float=0.15, lr:float=0.01, num_epochs:int=100):
#         # get dataloaders
#         train_loader, test_loader = self._get_dataloaders(self.dataset, batch_size, test_size)

#         # train model
#         optimizer = torch.optim.Adam(model.parameters(), lr=lr)
#         criterion = torch.nn.CrossEntropyLoss()

#         for epoch in range(1, num_epochs):
#             self._train(model, train_loader, optimizer, criterion)
#             train_acc = self._test(model, train_loader)
#             test_acc = self._test(model, test_loader)
#             print(f'Epoch: {epoch:03d}, Train Acc: {train_acc:.8f}, Test Acc: {test_acc:.8f}')

In [8]:
# trainer = QuickTrainer(
#     dataset=brca_pyg,
#     generator=generator
# )

# databatch = trainer.get_toy_databatch()